# Exercise 5.4 | Model Comparison with the F-Test

In Exercise 5.3, we built models of Pittsburgh housing prices with and without controls. Now lets test whether those controls actually improve the model, or whether we're just overfitting.

In [1]:
# Imports
import pandas as pd
import statsmodels.formula.api as smf
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

# File Path
file_path = 'https://econ-0150.tayweid.io/data/'

## Step 1: Load the Data and Fit Two Models

Load the Pittsburgh housing data and create the log price variable. Then fit two models:
- **Restricted**: `LOGPRICE ~ FINISHEDLIVINGAREA`
- **Full**: `LOGPRICE ~ FINISHEDLIVINGAREA + YEARBLT`

In [2]:
# Load data
data = pd.read_csv(file_path + 'pgh_housing.csv', index_col=0)
data['LOGPRICE'] = np.log(data['SALEPRICE'])
data.head()

,PARID,PROPERTYZIP,SALEDATE,SALEPRICE,STORIES,YEARBLT,TOTALROOMS,BEDROOMS,FULLBATHS,HALFBATHS,FINISHEDLIVINGAREA,LOGPRICE,AREA100
296306,1202H00136000000,15143,9/10/13,530960.0,2.0,2013.0,11.0,5.0,3.0,1.0,4435.0,13.182442,44.35
125627,0382E00004000000,15132,12/31/79,3500.0,2.0,1925.0,8.0,4.0,2.0,0.0,2228.0,8.160518,22.28
45045,0088S00003000000,15217,11/21/14,655000.0,2.0,2014.0,6.0,3.0,2.0,1.0,2976.0,13.392391,29.76
171940,0299B00100000000,15221,5/31/18,200000.0,2.0,1928.0,6.0,3.0,1.0,1.0,2218.0,12.206073,22.18
265608,0316R00212000000,15236,10/15/87,52000.0,1.0,1954.0,5.0,3.0,1.0,0.0,900.0,10.858999,9.00


In [3]:
# Restricted model: living area only
restricted = smf.ols('LOGPRICE ~ FINISHEDLIVINGAREA', data=data).fit()
print(restricted.summary().tables[1])

                         coef    std err          t      P>|t|      [0.025      0.975]
--------------------------------------------------------------------------------------
Intercept             10.2029      0.031    328.476      0.000      10.142      10.264
FINISHEDLIVINGAREA     0.0005   1.65e-05     33.260      0.000       0.001       0.001


In [4]:
# Full model: living area + year built
full = smf.ols('LOGPRICE ~ FINISHEDLIVINGAREA + YEARBLT', data=data).fit()
print(full.summary().tables[1])

                         coef    std err          t      P>|t|      [0.025      0.975]
--------------------------------------------------------------------------------------
Intercept            -17.9337      0.741    -24.213      0.000     -19.386     -16.482
FINISHEDLIVINGAREA     0.0005   1.56e-05     29.095      0.000       0.000       0.000
YEARBLT                0.0145      0.000     38.018      0.000       0.014       0.015


## Step 2: Compare R²

Use `model.rsquared` to get the R² for each model. How much did R² improve?

In [5]:
# Print R² for both models
print(f"Restricted R²: {restricted.rsquared:.4f}")
print(f"Full R²:       {full.rsquared:.4f}")

Restricted R²: 0.0996
Full R²:       0.2134


In [6]:
# Compute the improvement
print(f"Improvement:   {full.rsquared - restricted.rsquared:.4f}")

Improvement:   0.1137


## Step 3: Is the Improvement Real?

R² always goes up when you add variables. Use the F-test to check whether this improvement is statistically significant.

Use `full_model.compare_f_test(restricted_model)` to get the F-statistic and p-value.

In [7]:
# F-test: does adding year built significantly improve the model?
f_stat, p_value, df_diff = full.compare_f_test(restricted)
print(f"F-statistic: {f_stat:.2f}")
print(f"p-value:     {p_value:.6f}")

F-statistic: 1445.34
p-value:     0.000000


**Q.** What is the p-value? What does it tell us about adding year built to the model?

{++The p-value is 0.000000. Adding year built significantly improves the model.++}

## Step 4: Now Test a Noise Variable

Create a column of random noise and add it to the full model. Then test whether noise improves the model.

In [8]:
# Add random noise to the data
np.random.seed(42)
data['noise'] = np.random.normal(0, 1, len(data))

In [9]:
# Fit a model with noise added
noise_model = smf.ols('LOGPRICE ~ FINISHEDLIVINGAREA + YEARBLT + noise', data=data).fit()
print(noise_model.summary().tables[1])

                         coef    std err          t      P>|t|      [0.025      0.975]
--------------------------------------------------------------------------------------
Intercept            -17.9348      0.741    -24.211      0.000     -19.387     -16.483
FINISHEDLIVINGAREA     0.0005   1.56e-05     29.094      0.000       0.000       0.000
YEARBLT                0.0145      0.000     38.014      0.000       0.014       0.015
noise                 -0.0013      0.012     -0.111      0.912      -0.025       0.022


In [10]:
# Compare R²: did it go up?
print(f"Full R²:       {full.rsquared:.4f}")
print(f"With noise R²: {noise_model.rsquared:.4f}")
print(f"Improvement:   {noise_model.rsquared - full.rsquared:.4f}")

Full R²:       0.2134
With noise R²: 0.2134
Improvement:   0.0000


In [11]:
# F-test: does adding noise significantly improve the model?
f_stat_noise, p_value_noise, df_diff_noise = noise_model.compare_f_test(full)
print(f"F-statistic: {f_stat_noise:.2f}")
print(f"p-value:     {p_value_noise:.3f}")

F-statistic: 0.01
p-value:     0.912


**Q.** R² went up when you added noise. But what does the F-test say? What does this tell us about overfitting?

{++The F-statistic is 0.01 and the p-value is 0.912.++} Large p-value. The improvement is just overfitting.

## Step 5: Compare the Two Tests

Fill in the table with your results:

| Variable added | $\Delta R^2$ | F-statistic | p-value | Significant? |
|:---------------|:------------|:------------|:--------|:-------------|
| Year built     | {++0.1137++} | {++1445.34++} | {++0.000000++} | {++Yes++} |
| Random noise   | {++0.0000++} | {++0.01++} | {++0.912++} | {++No++} |

**Q.** Both variables increased R². Why did only one pass the F-test?

{++The F-test measures++} the size of the R² improvement relative to noise. {++The improvement from year built is much larger than noise would produce. The improvement from random noise is not.++}